<a href="https://colab.research.google.com/github/xinxingwu-uk/xinxingwu-uk/blob/main/LightGCN_Rating_Sentiment_MultiTask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Keras LightGCN 多任务：评分预测 + 文本情感辅助（Colab 可运行）

**目标**：在用户–商品二分图上，用 LightGCN 思想做 **评分回归（1–5 星）**，并引入 **评论文本情感**作为辅助任务（多任务学习）以提升效果。

你可以将此 Notebook 直接替换为 Amazon Reviews 的真实数据（`customer_id, product_id, review_text, rating`）。

## 安装依赖（在 Colab 上执行）

In [ ]:
!pip -q install tensorflow>=2.12.0
!pip -q install scikit-learn


## 1) 生成合成数据（可替换为真实 Amazon 数据）

In [ ]:
import math, numpy as np, random
import tensorflow as tf
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
random.seed(42)

num_users = 1000
num_items = 2000
embed_dim_true = 16

# 潜因子用于"制造"评分（训练时不会用到这些真值）
true_user = rng.normal(size=(num_users, embed_dim_true)).astype(np.float32)
true_item = rng.normal(size=(num_items, embed_dim_true)).astype(np.float32)
user_bias = rng.normal(scale=0.3, size=(num_users,)).astype(np.float32)
item_bias = rng.normal(scale=0.3, size=(num_items,)).astype(np.float32)
global_bias = 3.6

num_interactions = 100_000
user_ids = rng.integers(0, num_users, size=(num_interactions,), endpoint=False)
item_ids = rng.integers(0, num_items, size=(num_interactions,), endpoint=False)

dot = np.sum(true_user[user_ids] * true_item[item_ids], axis=1)
scores = (global_bias
          + 0.8 * dot / math.sqrt(embed_dim_true)
          + user_bias[user_ids] + item_bias[item_ids])
scores += rng.normal(scale=0.6, size=scores.shape)
ratings = np.clip(np.round(scores), 1, 5).astype(np.float32)

# 生成合成评论文本，并给出情感标签（-1/0/1）与 one-hot
pos_templates = [
    "Absolutely love it, works great!",
    "Five stars, highly recommended.",
    "Excellent quality and value.",
    "Very satisfied, will buy again.",
]
neu_templates = [
    "It's okay, nothing special.",
    "Average product, as expected.",
    "Neutral experience overall.",
    "Works fine for basic needs.",
]
neg_templates = [
    "Very disappointed, poor quality.",
    "Not worth the price.",
    "Bad experience, would not recommend.",
    "Stopped working after a week.",
]

def make_review_and_sentiment(r):
    # 评分越高，越可能正向；中间分数更可能中性；低分更可能负向
    if r >= 4.0:
        tpl = random.choice(pos_templates + neu_templates)
    elif r <= 2.0:
        tpl = random.choice(neg_templates + neu_templates)
    else:
        tpl = random.choice(neu_templates + pos_templates + neg_templates)
    # 简单启发式生成情感标签 -1/0/1
    if tpl in pos_templates: s = 1
    elif tpl in neg_templates: s = -1
    else: s = 0
    return tpl, s

reviews = []
sentis = []
for r in ratings:
    txt, s = make_review_and_sentiment(r)
    reviews.append(txt)
    sentis.append(s)
reviews = np.array(reviews)
sentis = np.array(sentis, dtype=np.int32)

# 划分：先打乱索引，然后 80/10/10
idx = np.arange(num_interactions)
rng.shuffle(idx)
train_end = int(0.8 * num_interactions)
val_end = int(0.9 * num_interactions)
train_idx, val_idx, test_idx = idx[:train_end], idx[train_end:val_end], idx[val_end:]

u_train, i_train, y_train = user_ids[train_idx], item_ids[train_idx], ratings[train_idx]
u_val,   i_val,   y_val   = user_ids[val_idx],   item_ids[val_idx],   ratings[val_idx]
u_test,  i_test,  y_test  = user_ids[test_idx],  item_ids[test_idx],  ratings[test_idx]

r_train, r_val, r_test = reviews[train_idx], reviews[val_idx], reviews[test_idx]
s_train, s_val, s_test = sentis[train_idx],  sentis[val_idx],  sentis[test_idx]

# 情感做 3 类分类（-1,0,1）→ 索引化到 {0,1,2}
s_train3 = (s_train + 1).astype(np.int32)
s_val3   = (s_val   + 1).astype(np.int32)
s_test3  = (s_test  + 1).astype(np.int32)


## 2) 构建训练图（仅用训练集边）与对称归一化邻接

In [ ]:
N = num_users + num_items
row = np.concatenate([u_train, i_train + num_users])
col = np.concatenate([i_train + num_users, u_train])

deg = np.zeros((N,), dtype=np.float32)
for r, c in zip(row, col):
    deg[r] += 1.0

weights = np.empty_like(row, dtype=np.float32)
for k, (r, c) in enumerate(zip(row, col)):
    di = max(deg[r], 1.0)
    dj = max(deg[c], 1.0)
    weights[k] = 1.0 / math.sqrt(di * dj)

indices = np.stack([row, col], axis=1).astype(np.int64)
A_hat = tf.sparse.reorder(tf.sparse.SparseTensor(indices=indices,
                                                 values=weights,
                                                 dense_shape=(N, N)))


## 3) 定义多任务 LightGCN 模型
- 基座：LightGCN 传播，得到所有节点（用户+商品）的表征。
- Pair 表征：取 `u_vec, i_vec` 并构造 `h_pair = [u, i, u*i, |u-i|]`。
- 文本表征：`TextVectorization + Embedding + GlobalAveragePooling`。
- 任务：
  - **评分回归**（MSE，输出 [1,5]）
  - **情感分类**（3 类，CrossEntropy）
- 总损失：`L = L_mse + λ * L_ce`，默认 λ=0.5

In [ ]:
from tensorflow.keras import layers, Model

class LightGCNBase(tf.keras.Model):
    def __init__(self, num_users, num_items, embed_dim=32, K=3, **kwargs):
        super().__init__(**kwargs)
        self.num_users = num_users
        self.num_items = num_items
        self.N = num_users + num_items
        self.K = K
        self.node_embeddings = tf.Variable(
            tf.keras.initializers.RandomNormal(stddev=0.1)(shape=(self.N, embed_dim)),
            name="node_embeddings"
        )
        self.beta = tf.Variable(0.0, name="beta")

    def call(self, A_norm, training=False):
        Xk = self.node_embeddings
        X_list = [Xk]
        for _ in range(self.K):
            Xk = tf.sparse.sparse_dense_matmul(A_norm, Xk)
            X_list.append(Xk)
        Xgcn = tf.add_n(X_list) / float(len(X_list))
        return Xgcn  # (N, d)

class MultiTaskLightGCN(Model):
    def __init__(self, num_users, num_items, embed_dim=32, K=3, lambda_ce=0.5):
        super().__init__()
        self.base = LightGCNBase(num_users, num_items, embed_dim=embed_dim, K=K)
        self.num_users = num_users
        self.lambda_ce = lambda_ce

        # 文本塔
        self.vectorize = layers.TextVectorization(max_tokens=20000, output_sequence_length=40)
        self.text_emb = layers.Embedding(input_dim=20000, output_dim=64)
        self.text_pool = layers.GlobalAveragePooling1D()

        # 评分头
        self.score_mlp = tf.keras.Sequential([
            layers.Dense(64, activation='relu'),
            layers.Dense(1)  # 先输出实数，后续映射到 [1,5]
        ])

        # 情感头（3 类：neg/neu/pos）
        self.senti_mlp = tf.keras.Sequential([
            layers.Dense(64, activation='relu'),
            layers.Dense(3)  # logits
        ])

    def adapt_text(self, texts):
        self.vectorize.adapt(texts)

    def call(self, inputs, training=False):
        user_id, item_id, A_norm, review_txt = inputs
        Xgcn = self.base(A_norm, training=training)
        u_vec = tf.gather(Xgcn, user_id)
        i_vec = tf.gather(Xgcn, item_id + self.base.num_users)
        pair = tf.concat([u_vec, i_vec, u_vec * i_vec, tf.abs(u_vec - i_vec)], axis=-1)

        # 文本表征
        tkn = self.vectorize(review_txt)
        txt_vec = self.text_emb(tkn)
        txt_vec = self.text_pool(txt_vec)

        h = tf.concat([pair, txt_vec], axis=-1)

        # 评分预测：实数 -> Sigmoid -> [1,5]
        raw = self.score_mlp(h)
        score_pred = 1.0 + 4.0 * tf.sigmoid(raw)

        # 情感预测：3 类 logits
        senti_logits = self.senti_mlp(h)
        return score_pred, senti_logits

    def compute_loss(self, data, y_true):
        # y_true = (rating_float, senti_int)
        (user_id, item_id, A_norm, review_txt) = data
        y_rating, y_senti = y_true
        y_pred_rating, y_pred_senti_logits = self([user_id, item_id, A_norm, review_txt], training=True)
        mse = tf.reduce_mean(tf.square(y_pred_rating[:,0] - y_rating))
        ce  = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=y_senti, logits=y_pred_senti_logits))
        return mse + self.lambda_ce * ce, mse, ce


## 4) 训练与评估

In [ ]:
batch_size = 2048
epochs = 5
lambda_ce = 0.5

model = MultiTaskLightGCN(num_users, num_items, embed_dim=32, K=3, lambda_ce=lambda_ce)
model.adapt_text(r_train)

A_const = A_hat  # 作为常量捕获

def make_ds(u, i, r, s3, txt, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(((u, i, txt), (r, s3)))
    if shuffle:
        ds = ds.shuffle(8192, seed=42, reshuffle_each_iteration=True)
    def _map(u, i, txt, y):
        # 把稀疏邻接注入
        return (u, i, A_const, txt), y
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(u_train, i_train, y_train, s_train3, r_train, shuffle=True)
val_ds   = make_ds(u_val,   i_val,   y_val,   s_val3,   r_val)
test_ds  = make_ds(u_test,  i_test,  y_test,  s_test3,  r_test)

opt = tf.keras.optimizers.Adam(1e-3)

train_logs = []
for epoch in range(1, epochs+1):
    # 训练
    tr_loss = tr_mse = tr_ce = 0.0
    steps = 0
    for (xb, yb) in train_ds:
        with tf.GradientTape() as tape:
            loss, mse, ce = model.compute_loss(xb, yb)
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        tr_loss += float(loss); tr_mse += float(mse); tr_ce += float(ce); steps += 1
    tr_loss/=steps; tr_mse/=steps; tr_ce/=steps

    # 验证
    val_mse = val_ce = 0.0
    vsteps = 0
    for (xb, yb) in val_ds:
        loss, mse, ce = model.compute_loss(xb, yb)
        val_mse += float(mse); val_ce += float(ce); vsteps += 1
    val_mse/=vsteps; val_ce/=vsteps

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} (mse={tr_mse:.4f}, ce={tr_ce:.4f}) | val_mse={val_mse:.4f} val_ce={val_ce:.4f}")
    train_logs.append((epoch, tr_loss, tr_mse, tr_ce, val_mse, val_ce))

# 测试集评估：RMSE + 情感准确率
import math
se_sum = 0.0
n = 0
correct = 0
n_cls = 0
for (xb, yb) in test_ds:
    (u_b, i_b, A_b, txt_b), (r_b, s_b) = xb, yb
    pred_r, pred_slogits = model([u_b, i_b, A_b, txt_b], training=False)
    pred_r = tf.squeeze(pred_r, -1)
    se_sum += float(tf.reduce_sum(tf.square(pred_r - r_b)))
    n += int(r_b.shape[0])
    pred_s = tf.argmax(pred_slogits, axis=-1, output_type=tf.int32)
    correct += int(tf.reduce_sum(tf.cast(pred_s == s_b, tf.int32)))
    n_cls += int(s_b.shape[0])

rmse = math.sqrt(se_sum / max(n,1))
acc = correct / max(n_cls,1)
print(f"Test RMSE={rmse:.4f}, Sentiment Acc={acc:.4f}")


## 5) 用真实 Amazon 数据时如何替换
1. 读取你的 CSV/Parquet：得到列 `customer_id, product_id, review_text, rating`。
2. 将 `customer_id` 与 `product_id` **编码为连续索引** `[0,U), [0,I)`；评分转为 `float32`。
3. 按时间做 **train/val/test** 切分（避免未来泄漏）。
4. 用真实的 `(user_ids, item_ids, reviews, ratings)` 替换本 Notebook 的合成数据段，其它代码保持不变。
5. 如需中文情感，可换更强的文本塔（例如导入外部中文词表/预训练词向量或接驳轻量 Transformer）。

## 6) 原理补充：为什么推荐/评分预测天然是二分图？

在推荐系统里，**用户 (U)** 和 **商品 (I)** 是两类不同实体，交互（点击/加购/购买/评论/评分）构成 **U–I 的边**。这张图：

- **节点**：两类（异质）——用户节点和商品节点；
- **边**：用户与商品之间的交互（可以带权重，如次数、停留时长、评分、时间衰减等）；
- **图结构**：**二分图 (Bipartite Graph)**，即图的节点集合可被分成两部分，边只跨两部分相连。

这样建模的好处：
1. **结构先验**：用户与商品本来就不属于同一类型实体，二分图恰好匹配；
2. **信息传播**：通过图传播（如 LightGCN 的 \(\hat{A}X\)），用户节点可以“吸收”邻接商品的特征分布，商品节点也能从消费它的用户群得到“画像”；
3. **可扩展性**：边可不断追加（新交互、时间衰减），图学习迭代更新；
4. **任务对齐**：
   - **评分预测/链接回归**：对给定 (u,i) 预测连续评分；
   - **召回/排序**：预测 (u,i) 的匹配度作为排序分数；
   - **多任务**：在同一对 (u,i) 上同时预测多种目标（如评分+情感+是否复购），共享图表示提升鲁棒性。

与“给每个用户单独建图做图分类”相比，二分图更**直接对应**“用户×商品”的预测目标（边上的数值/概率），同时自然支持大规模在线服务。